# Chapter 19 - Training G-Retriever and a LoRA baseline on ExplaGraphs

Hands-On Graph Neural Networks Using Python, Second Edition.

This notebook trains two models with the same data, seed, and hyperparameters:

1. **G-Retriever**: a GAT encoder that uses both node and relation embeddings, plus Qwen3-0.6B with LoRA.
2. **LoRA baseline**: the same Qwen3-0.6B with LoRA and the same linearized graph in the prompt, but no GNN.

It requires a CUDA GPU runtime (Runtime > Change runtime type). Run the cells in order. The last cell downloads the trained weights of both models (`gretriever_explagraphs.pt` and `lora_explagraphs.pt`). Place them in the `checkpoints/` folder next to `run.py`, which evaluates both models on the test set, also on a CPU.

## 1. Install the dependencies and check the GPU

In [1]:
!pip install -q 'torch_geometric==2.8.0' transformers accelerate peft sentencepiece
!pip install -q -U torchao
!nvidia-smi

Tue Oct  6 12:55:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Setup and dataset loader

In [2]:
import csv, json, math, os, random, re, time, urllib.request
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.llm.models import LLM, GRetriever, SentenceTransformer
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GAT

SEED = 0
device = torch.device('cuda')
print(f'Device: {torch.cuda.get_device_name(0)}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_llm():
    llm = LLM(
        model_name='Qwen/Qwen3-0.6B',
        num_params=0.6,
        dtype=torch.bfloat16,
        sys_prompt=(
            '/no_think You are a helpful assistant. Given a belief, an '
            'argument, and a commonsense graph, decide whether the argument '
            'supports or counters the belief. Answer with a single word: '
            'support or counter.'
        ),
    )
    llm.llm.generation_config.do_sample = False
    llm.llm.generation_config.temperature = None
    llm.llm.generation_config.top_p = None
    llm.llm.generation_config.top_k = None
    return llm


DATA_DIR = './data/ExplaGraphs'
BASE = 'https://raw.githubusercontent.com/swarnaHub/ExplaGraphs/master/data'
os.makedirs(DATA_DIR, exist_ok=True)


def download_files():
    for name in ['train.tsv', 'dev.tsv']:
        local = os.path.join(DATA_DIR, name)
        if not os.path.exists(local):
            urllib.request.urlretrieve(f'{BASE}/{name}', local)


def parse_graph(graph_str):
    triples = []
    for match in re.findall(r'\(([^)]+)\)', graph_str):
        parts = [p.strip() for p in match.split(';')]
        if len(parts) == 3:
            triples.append(tuple(parts))
    return triples


def read_tsv(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        for r in reader:
            if len(r) < 4:
                continue
            belief, argument, stance, graph_str = r[:4]
            if stance not in ('support', 'counter'):
                continue
            triples = parse_graph(graph_str)
            if not triples:
                continue
            rows.append({'belief': belief.strip(),
                         'argument': argument.strip(),
                         'stance': stance.strip(),
                         'triples': triples})
    return rows


def build_dataset(rows, sbert):
    dataset = []
    for row in rows:
        nodes, node_ids, es, ed, er = [], {}, [], [], []
        for src, rel, dst in row['triples']:
            for txt in (src, dst):
                if txt not in node_ids:
                    node_ids[txt] = len(nodes)
                    nodes.append(txt)
            es.append(node_ids[src])
            ed.append(node_ids[dst])
            er.append(rel)

        x = sbert.encode(nodes, batch_size=64, output_device='cpu')
        edge_attr = sbert.encode(er, batch_size=64, output_device='cpu')

        s = torch.tensor(es, dtype=torch.long)
        d = torch.tensor(ed, dtype=torch.long)
        edge_index = torch.stack([torch.cat([s, d]), torch.cat([d, s])], dim=0)
        edge_attr = torch.cat([edge_attr, edge_attr], dim=0)

        desc = ' '.join(f'({s}; {r}; {d})' for s, r, d in row['triples'])
        question = (f"/no_think Belief: {row['belief']}\n"
                    f"Argument: {row['argument']}\n"
                    f"Does the argument support or counter the belief? "
                    f"Answer with a single word: support or counter.")

        dataset.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                            question=question, label=row['stance'], desc=desc))
    return dataset

Device: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3. Build the dataset

In [3]:
set_seed(SEED)
download_files()
sbert = SentenceTransformer(
    'sentence-transformers/all-roberta-large-v1'
).to(device)
sbert.eval()

train_rows = read_tsv(os.path.join(DATA_DIR, 'train.tsv'))
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(train_rows))
split = int(0.8 * len(train_rows))
train_ds = build_dataset([train_rows[i] for i in idx[:split]], sbert)
val_ds = build_dataset([train_rows[i] for i in idx[split:]], sbert)

print(f'train={len(train_ds)}, val={len(val_ds)}')
support = sum(d.label == 'support' for d in val_ds)
print(f'val majority class (support): {support/len(val_ds):.3f}')

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

train=1894, val=474
val majority class (support): 0.584


## 4. Training function

In [4]:
LR, WD, MIN_LR, GRAD_CLIP = 1e-5, 0.05, 5e-6, 0.1
NUM_EPOCHS, WARMUP_EPOCHS = 10, 1
os.makedirs('./checkpoints', exist_ok=True)


def adjust_lr(pg, epoch, total):
    if epoch < WARMUP_EPOCHS:
        lr = LR
    else:
        prog = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
        lr = MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * prog))
    pg['lr'] = lr
    return lr


def first_word(pred):
    answer = re.sub(r'<think>.*?</think>', '', pred, flags=re.DOTALL)
    words = answer.strip().lower().split()
    return re.sub(r'[^a-z]', '', words[0]) if words else ''


def is_correct(pred, label):
    return first_word(pred).startswith(label[:4].lower())


def compute_loss(model, batch):
    batch = batch.to(device)
    return model(
        question=batch.question,
        x=batch.x.float(),
        edge_index=batch.edge_index,
        batch=batch.batch,
        label=batch.label,
        edge_attr=batch.edge_attr.float(),
        additional_text_context=batch.desc,
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, n_seen, correct = 0.0, 0, 0
    for batch in loader:
        loss = compute_loss(model, batch)
        total_loss += loss.item() * batch.num_graphs
        n_seen += batch.num_graphs
        preds = model.inference(
            question=batch.question,
            x=batch.x.float().to(device),
            edge_index=batch.edge_index.to(device),
            batch=batch.batch.to(device),
            edge_attr=batch.edge_attr.float().to(device),
            additional_text_context=batch.desc,
            max_out_tokens=24,
        )
        correct += sum(is_correct(p, l) for p, l in zip(preds, batch.label))
    model.train()
    return total_loss / n_seen, correct / n_seen


def train(model, name):
    set_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WD,
    )
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    for epoch in range(NUM_EPOCHS):
        lr = adjust_lr(optimizer.param_groups[0], epoch, NUM_EPOCHS)
        model.train()
        train_loss, n_seen = 0.0, 0
        for batch in train_loader:
            optimizer.zero_grad()
            loss = compute_loss(model, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item() * batch.num_graphs
            n_seen += batch.num_graphs
        train_loss /= n_seen
        val_loss, val_acc = evaluate(model, val_loader)
        print(f'--- Epoch {epoch+1}/{NUM_EPOCHS} ---')
        print(f'  lr={lr:.2e}  train_loss={train_loss:.4f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

    minutes = (time.time() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f'Training time: {minutes:.1f} min | Peak GPU memory: {peak:.1f} GB')

    state = {k: v.detach().cpu()
             for k, v in model.named_parameters() if v.requires_grad}
    path = f'./checkpoints/{name}_explagraphs.pt'
    torch.save({'epoch': NUM_EPOCHS, 'state_dict': state, 'val_acc': val_acc},
               path)
    print(f'Saved {path} ({os.path.getsize(path)/1e6:.1f} MB)')

## 5. Train G-Retriever (GNN with relation embeddings)

In [5]:
set_seed(SEED)
gnn = GAT(
    in_channels=1024,
    hidden_channels=1024,
    num_layers=4,
    out_channels=1024,
    heads=4,
    edge_dim=1024,
)
model = GRetriever(
    llm=make_llm(),
    gnn=gnn,
    use_lora=True,
    mlp_out_tokens=1,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.1f}M '
      f'({100*trainable/total:.2f}%)')

train(model, 'gretriever')

Setting up 'Qwen/Qwen3-0.6B' with configuration: {'revision': 'main', 'max_memory': {0: '92GiB'}, 'low_cpu_mem_usage': True, 'device_map': 'auto', 'dtype': torch.bfloat16}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Trainable parameters: 18.0M / 614.0M (2.92%)
--- Epoch 1/10 ---
  lr=1.00e-05  train_loss=1.4442  val_loss=0.3835  val_acc=0.584
--- Epoch 2/10 ---
  lr=1.00e-05  train_loss=0.3076  val_loss=0.3523  val_acc=0.783
--- Epoch 3/10 ---
  lr=9.85e-06  train_loss=0.3386  val_loss=0.5203  val_acc=0.829
--- Epoch 4/10 ---
  lr=9.42e-06  train_loss=0.3128  val_loss=0.4372  val_acc=0.857
--- Epoch 5/10 ---
  lr=8.75e-06  train_loss=0.2654  val_loss=0.4593  val_acc=0.867
--- Epoch 6/10 ---
  lr=7.93e-06  train_loss=0.2352  val_loss=0.4343  val_acc=0.873
--- Epoch 7/10 ---
  lr=7.07e-06  train_loss=0.1852  val_loss=0.4607  val_acc=0.865
--- Epoch 8/10 ---
  lr=6.25e-06  train_loss=0.1647  val_loss=0.6011  val_acc=0.857
--- Epoch 9/10 ---
  lr=5.58e-06  train_loss=0.1501  val_loss=0.5407  val_acc=0.867
--- Epoch 10/10 ---
  lr=5.15e-06  train_loss=0.1262  val_loss=0.5766  val_acc=0.863
Training time: 5.6 min | Peak GPU memory: 8.5 GB
Saved ./checkpoints/gretriever_explagraphs.pt (71.9 MB)


## 6. Train the LoRA baseline (same LLM and prompt, no GNN)

In [6]:
set_seed(SEED)
baseline = GRetriever(
    llm=make_llm(),
    gnn=None,
    use_lora=True,
).to(device)

trainable = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
total = sum(p.numel() for p in baseline.parameters())
print(f'Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.1f}M '
      f'({100*trainable/total:.2f}%)')

train(baseline, 'lora')

Setting up 'Qwen/Qwen3-0.6B' with configuration: {'revision': 'main', 'max_memory': {0: '90GiB'}, 'low_cpu_mem_usage': True, 'device_map': 'auto', 'dtype': torch.bfloat16}


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Trainable parameters: 1.1M / 597.2M (0.19%)
--- Epoch 1/10 ---
  lr=1.00e-05  train_loss=2.3679  val_loss=0.3251  val_acc=0.605
--- Epoch 2/10 ---
  lr=1.00e-05  train_loss=0.2868  val_loss=0.3524  val_acc=0.749
--- Epoch 3/10 ---
  lr=9.85e-06  train_loss=0.3005  val_loss=0.3961  val_acc=0.838
--- Epoch 4/10 ---
  lr=9.42e-06  train_loss=0.3148  val_loss=0.4388  val_acc=0.850
--- Epoch 5/10 ---
  lr=8.75e-06  train_loss=0.2989  val_loss=0.3857  val_acc=0.869
--- Epoch 6/10 ---
  lr=7.93e-06  train_loss=0.2646  val_loss=0.3877  val_acc=0.871
--- Epoch 7/10 ---
  lr=7.07e-06  train_loss=0.2229  val_loss=0.3726  val_acc=0.869
--- Epoch 8/10 ---
  lr=6.25e-06  train_loss=0.2089  val_loss=0.3923  val_acc=0.888
--- Epoch 9/10 ---
  lr=5.58e-06  train_loss=0.1812  val_loss=0.4344  val_acc=0.869
--- Epoch 10/10 ---
  lr=5.15e-06  train_loss=0.1645  val_loss=0.4249  val_acc=0.873
Training time: 5.1 min | Peak GPU memory: 10.8 GB
Saved ./checkpoints/lora_explagraphs.pt (4.6 MB)


## 7. Download the trained weights

In [7]:
from google.colab import files

files.download('./checkpoints/gretriever_explagraphs.pt')
files.download('./checkpoints/lora_explagraphs.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>